In [0]:
import os
from pyspark.sql.functions import current_timestamp, from_utc_timestamp

# ORIGEM: Volume de landing zone
volume_origem = "/Volumes/workspace/yelp_ing/landing_zone_kaggle_api/"

# DESTINO: Catalog e Schema
catalog_destino = "workspace"
schema_destino = "yelp_bronze"

# Lista todos os arquivos no volume de origem
files = [f.path for f in dbutils.fs.ls(volume_origem) if not f.isDir()]

for file_path in files:
    # Extrai o nome do arquivo sem extensão
    file_name = os.path.basename(file_path)
    table_name_simples = f"{os.path.splitext(file_name)[0]}"
    
    # Nome completo da tabela com catalog e schema
    table_name_completo = f"{catalog_destino}.{schema_destino}.{table_name_simples}"
    
    # Inferência do formato pelo sufixo do arquivo
    if file_name.endswith(".csv"):
        df = spark.read.option("header", "true").csv(file_path)
    elif file_name.endswith(".json"):
        df = spark.read.json(file_path)
    elif file_name.endswith(".parquet"):
        df = spark.read.parquet(file_path)
    else:
        print(f"Formato não suportado para o arquivo: {file_name}")
        continue
    
    # Adiciona coluna com timestamp da ingestão no horário de Brasília
    df = df.withColumn("hora_ingestao", from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))
    
    # Cria a tabela bronze no schema YELP_BRONZE
    df.write.mode("overwrite").saveAsTable(table_name_completo)
    df.write.mode("append").saveAsTable(table_name_completo + "_hist")
    print(f"✓ Tabela criada/atualizada: {table_name_completo}")
    print(f"✓ Tabela criada/atualizada: {table_name_completo}_hist")

In [0]:
%sql
-- Testando append da tabela histórica - count normal
select count(*) from  workspace.yelp_bronze.yelp_academic_dataset_business

In [0]:
%sql
-- Testando append da tabela histórica - count hist
select count(*) from  workspace.yelp_bronze.yelp_academic_dataset_business_hist

In [0]:
#%python
#import os
#
## Caminho do volume
#volume_path = "/Volumes/workspace/yelp_ing/landing_zone_kaggle_api/"
#
## Lista todos os arquivos no volume
#files = [f.path for f in dbutils.fs.ls(volume_path) if not f.isDir()]
#
#for file_path in files:
#    # Extrai o nome do arquivo sem extensão para usar como nome da tabela
#    file_name = os.path.basename(file_path)
#    table_name = f"bronze_{os.path.splitext(file_name)[0]}"
#    
#    # Inferência simples do formato pelo sufixo do arquivo
#    if file_name.endswith(".csv"):
#        df = spark.read.option("header", "true").csv(file_path)
#    elif file_name.endswith(".json"):
#        df = spark.read.json(file_path)
#    elif file_name.endswith(".parquet"):
#        df = spark.read.parquet(file_path)
#    else:
#        print(f"Formato não suportado para o arquivo: {file_name}")
#        continue
#    
#    # Cria a tabela bronze (no schema atual)
#    df.write.mode("overwrite").saveAsTable(table_name)
#    print(f"Tabela criada: {table_name}")

In [0]:
## ATENÇÃO: Este código vai dropar TODAS as tabelas do schema yelp_ing
#
#catalog = "workspace"
#schema = "yelp_ing"
#
## Lista todas as tabelas no schema
#tabelas = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()
#
#if len(tabelas) == 0:
#    print(f"Nenhuma tabela encontrada no schema {catalog}.{schema}")
#else:
#    print(f"Dropando {len(tabelas)} tabela(s) do schema {catalog}.{schema}...\n")
#    
#    for row in tabelas:
#        table_name = row.tableName
#        full_table_name = f"{catalog}.{schema}.{table_name}"
#        
#        try:
#            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
#            print(f"✓ Tabela dropada: {full_table_name}")
#        except Exception as e:
#            print(f"✗ Erro ao dropar {full_table_name}: {str(e)}")
#    
#    print(f"\nProcesso concluído!")

In [0]:
#spark.sql("drop table workspace.yelp_ing.bronze_yelp_academic_dataset_user")